# EMNIST: эксперименты с тремя новыми архитектурами

**Сетка:** 3 архитектуры (v4, v5, v7) × 3 seed (42, 123, 7) = **9 запусков**

**Параметры:** 25 эпох, lr=1e-3, batch=64, аугментация=light, scheduler=cosine, AdamW, label_smoothing=0.1

**Ожидаемое время:** ~3ч 45мин на T4.

## 1. Клонирование репозитория и установка зависимостей

In [ ]:
# === ЗАМЕНИ URL на свой репозиторий ===
REPO_URL = 'https://github.com/YOUR_USERNAME/emnist-project.git'

import os
if not os.path.exists('/content/emnist-project'):
    !git clone $REPO_URL /content/emnist-project
%cd /content/emnist-project

!pip install -q tensorboard seaborn

## 2. Проверка GPU

In [ ]:
import torch
print(f'CUDA доступна: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
!nvidia-smi

## 3. Скачивание EMNIST (если ещё не скачан)

Если в репозитории есть `scripts/download_emnist.py` — запусти его. Иначе используй стандартный загрузчик torchvision (он сработает в первом эксперименте автоматически).

In [ ]:
# Если есть скрипт:
# !python scripts/download_emnist.py

# Иначе автоматически скачается при первом запуске train.py

## 4. ⚠️ ВАЖНО: проверь, что в train.py применены патчи

Перед запуском в `src/training/train.py` должно быть:
1. `criterion = nn.CrossEntropyLoss(label_smoothing=0.1)`
2. `optimizer = optim.AdamW(...)`
3. В реестре моделей добавлены `improved_cnn_v5` и `improved_cnn_v7`
4. В `choices` парсера тоже добавлены v5 и v7

Также проверь, что в `src/models/` лежат файлы:
- `improved_cnn_v4.py`
- `improved_cnn_v5.py`
- `improved_cnn_v7.py`

In [ ]:
# Проверка наличия файлов
import os
for f in ['src/models/improved_cnn_v4.py',
          'src/models/improved_cnn_v5.py',
          'src/models/improved_cnn_v7.py']:
    status = '✓' if os.path.exists(f) else '✗ ОТСУТСТВУЕТ'
    print(f'{status} {f}')

## 5. Запуск всех 9 экспериментов

In [ ]:
import subprocess, time
from datetime import datetime

# === Конфигурация сетки ===
MODELS = ['improved_cnn_v4', 'improved_cnn_v5', 'improved_cnn_v7']
SEEDS  = [42, 123, 7]

# Фиксированные параметры (best-конфиг от v3)
EPOCHS       = 25
BATCH_SIZE   = 64
LR           = 1e-3
WEIGHT_DECAY = 1e-4
AUGMENTATION = 'light'
SCHEDULER    = 'cosine'
SPLIT        = 'balanced'

total = len(MODELS) * len(SEEDS)
print(f'Запусков: {total}')
print(f'Старт: {datetime.now().strftime("%H:%M:%S")}')
print('=' * 70)

t_start = time.time()
results = []

for i, model in enumerate(MODELS):
    for j, seed in enumerate(SEEDS):
        run_idx = i * len(SEEDS) + j + 1
        exp_name = f'{model}_seed{seed}_light_cosine'

        cmd = [
            'python', '-m', 'src.training.train',
            '--model', model,
            '--experiment', exp_name,
            '--epochs', str(EPOCHS),
            '--batch_size', str(BATCH_SIZE),
            '--lr', str(LR),
            '--weight_decay', str(WEIGHT_DECAY),
            '--augmentation', AUGMENTATION,
            '--scheduler', SCHEDULER,
            '--split', SPLIT,
            '--seed', str(seed),
            '--num_workers', '2',
        ]

        print(f'\n[{run_idx}/{total}] {exp_name}')
        print(f'  Старт: {datetime.now().strftime("%H:%M:%S")}')
        t0 = time.time()

        result = subprocess.run(cmd, capture_output=True, text=True)

        elapsed = time.time() - t0
        status = 'OK' if result.returncode == 0 else 'FAILED'
        print(f'  Статус: {status} | Время: {elapsed/60:.1f} мин')

        if result.returncode != 0:
            print('  Последние строки stderr:')
            print('  ' + '\n  '.join(result.stderr.strip().split('\n')[-5:]))

        results.append({
            'experiment': exp_name,
            'status': status,
            'time_min': elapsed / 60,
        })

print('\n' + '=' * 70)
total_min = (time.time() - t_start) / 60
print(f'ВСЕГО ВРЕМЕНИ: {total_min:.1f} мин ({total_min/60:.2f} ч)')
print(f'OK: {sum(1 for r in results if r["status"] == "OK")}/{total}')

## 6. Архивация результатов и выгрузка

In [ ]:
# Создаём архив только новых экспериментов (без .pth и event-файлов)
!cd /content/emnist-project && \
  tar --exclude='*.pth' --exclude='*.pt' \
      --exclude='events.out.tfevents.*' --exclude='tensorboard' \
      -czvf new_experiments.tar.gz \
      results/*v4* results/*v5* results/*v7* 2>&1 | tail -5

!ls -lh /content/emnist-project/new_experiments.tar.gz

In [ ]:
# Скачивание на компьютер
from google.colab import files
files.download('/content/emnist-project/new_experiments.tar.gz')

## 7. (Опционально) Сохранить в Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/emnist-project/new_experiments.tar.gz /content/drive/MyDrive/
print('Сохранено в Google Drive')